# Predição das imagens do Lote 4 (Revisadas) com o modelo MSO-V1

Autor:  Viviane da Rosa Sommer

Atualizado: 14/11/2024

## Objetivo:
Notebook para fazer a predição das imagens do Lote 4 (revisadas) pelo modelo MSO-V1, para analisar o desempenho dele em comparação com a marcação dos especialistas.

## Importações Necessárias

In [ ]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install torch

import glob
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os 
import torchvision
import json
import torch

## Declaração de Constantes e Modelos

In [ ]:
DUCT_CLASS_ID = 0
CORAL_CLASS_ID = 0
RESIZED_DIM_DUTO = 640
RESIZED_DIM_CORAL = 800

INPUT_DIRECTORY = "Lote 4 e 5/Imagens-Revisadas-Lote4"
IMAGE_DIRECTORY = "Lote 4 e 5/Imagens-Nao-Revisadas-Lote4e5/Revisão-Lotes4-e-5/Lote4-Revisao"
OUTPUT_DIRECTORY = "pred_mso_v1_lote_4_5_c_revisao"
os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)

easy_duct_model = YOLO('runs/segment/train-facil-500-v1/weights/best.pt')
medium_duct_model = YOLO('runs/segment/train-medio-500-v1/weights/best.pt')
hard_duct_model = YOLO('runs/segment/train-dificil-500-v1/weights/best.pt')
coral_model = YOLO('runs/segment/Lote123-SOverlay/train-800-lote123/weights/best.pt')

## Funções Necessárias

In [ ]:
def read_image(filename: str) -> tuple:
    """
    Reads an image from a specified file and returns the image along with its height and width.

    Parameters:
        filename (str): The path to the image file.

    Returns:
        tuple: A tuple containing:
            - img (np.ndarray): The loaded image in BGR format.
            - base_height (int): The height of the image.
            - base_width (int): The width of the image.
            If the image cannot be loaded, returns (None, None, None).
    """
    img = cv2.imread(filename)
    if img is None:
        print(f"Failed to load image: {filename}")
        return None, None, None
        
    base_height, base_width = img.shape[:2]
    return img, base_height, base_width


def prediction_duct(img: np.array) -> tuple:
    """
    Makes a prediction using the duct model and returns the mask for duct objects.

    Parameters:
        img (np.array): The input image for prediction.

    Returns:
        tuple: A tuple containing:
            - resized_duct_mask (np.ndarray): The resized mask for duct objects.
            - duct_mask_tensor (torch.Tensor): The tensor version of the duct mask.
    """
    
    original_height, original_width = img.shape[:2]
    best_result = None
    for model in [easy_duct_model, medium_duct_model, hard_duct_model]:
        duct_results = model(img, verbose=False)
        best_result = process_results(duct_results)
        if best_result is not None:
            break

    if best_result is not None and best_result.masks is not None:
        masks = best_result.masks.data
        boxes = best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]  

        duct_indices = torch.where(classes == DUCT_CLASS_ID)[0]
        duct_boxes = boxes[duct_indices]
        duct_masks = masks[duct_indices]

        if len(duct_boxes) > 0:
            nms_indices = torchvision.ops.nms(duct_boxes[:, :4], scores[duct_indices], iou_threshold=0.2)
            duct_boxes = duct_boxes[nms_indices]
            duct_masks = duct_masks[nms_indices]
            final_duct_mask = torch.any(duct_masks, dim=0).int() * 255
        else:
            final_duct_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
    else:
        final_duct_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
        
    final_duct_mask_np = final_duct_mask.cpu().numpy()
    resized_final_duct_mask = cv2.resize(final_duct_mask_np, (original_width, original_height), interpolation=cv2.INTER_NEAREST)
    return resized_final_duct_mask, torch.from_numpy(resized_final_duct_mask).int()


def process_results(results: list) -> any:
    """
    Processes the results obtained from the model and returns the first valid result.

    Parameters:
        results (list): List of model detection results.

    Returns:
        any: The first valid result containing masks, or None if no valid result is found.
    """
    for result in results:
        if result.masks is not None:
            return result
    return None


def generate_mask_from_polygon(polygon: list, height: int, width: int) -> np.ndarray:
    """
    Generates a binary mask from a specified polygon defined by its vertices.

    Parameters:
        polygon (list): A list of (x, y) coordinates defining the vertices of the polygon.
        height (int): The height of the mask to be generated.
        width (int): The width of the mask to be generated.

    Returns:
        np.ndarray: A binary mask of size (height, width), where the polygon area is filled with 1.
    """
    mask = np.zeros((height, width), dtype=np.uint8)
    polygon = np.array(polygon, dtype=np.int32)  
    cv2.fillPoly(mask, [polygon], 1) 
    return mask
                    
                    
def prediction_coral(img: np.array) -> tuple:
    """
    Makes a prediction using the coral model and returns a mask for coral objects.

    Parameters:
        img (np.array): The input image for prediction.

    Returns:
        tuple: A tuple containing:
            - centered_mask (np.ndarray): The centered mask for coral objects.
            - coral_mask_tensor (torch.Tensor): The tensor version of the coral mask.
    """
    original_height, original_width = img.shape[:2]
    coral_results = coral_model(img, verbose=False)
    coral_best_result = process_results(coral_results)
    
    if coral_best_result is not None and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        coral_boxes = boxes[coral_indices]
        coral_masks = masks[coral_indices]
        coral_scores = scores[coral_indices]

        if len(coral_boxes) > 0:
            nms_indices = torchvision.ops.nms(coral_boxes[:, :4], coral_scores, iou_threshold=0.2)
            coral_boxes = coral_boxes[nms_indices]
            coral_masks = coral_masks[nms_indices]
            final_coral_mask = torch.any(coral_masks, dim=0).int() * 255
        else:
            final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
    else:
        final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)

    final_coral_mask_np = final_coral_mask.cpu().numpy()
    mask_height, mask_width = final_coral_mask_np.shape

    resized_final_coral_mask = cv2.resize(final_coral_mask_np, (original_width, original_height), interpolation=cv2.INTER_NEAREST)
    centered_mask = np.zeros((original_height, original_width), dtype=np.uint8)
    y_offset = (original_height - resized_final_coral_mask.shape[0]) // 2
    x_offset = (original_width - resized_final_coral_mask.shape[1]) // 2
    if (y_offset >= 0 and x_offset >= 0 and 
        y_offset + resized_final_coral_mask.shape[0] <= original_height and 
        x_offset + resized_final_coral_mask.shape[1] <= original_width):
        
        centered_mask[y_offset:y_offset + resized_final_coral_mask.shape[0], 
                      x_offset:x_offset + resized_final_coral_mask.shape[1]] = resized_final_coral_mask
    else:
        centered_mask = resized_final_coral_mask.copy()

    return centered_mask, torch.from_numpy(centered_mask).int()


def read_annotation_file(filename: str) -> torch.Tensor:
    """
    Reads an annotation file and generates a binary mask for the coral object.

    Parameters:
        filename (str): The path to the annotation file.

    Returns:
        torch.Tensor: A binary mask of the coral annotation.
    """
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            annotation_data = json.load(f)
        annotated_coral_mask = torch.zeros((base_height, base_width), dtype=torch.uint8)
        for shape in annotation_data['shapes']:
            polygon = shape['points']
            mask = generate_mask_from_polygon(polygon, base_height, base_width)
            mask_tensor = torch.from_numpy(mask)
            annotated_coral_mask = annotated_coral_mask | mask_tensor
    else:
        annotated_coral_mask = torch.zeros((base_height, base_width), dtype=torch.uint8)
    annotated_coral_mask *= 255
    return annotated_coral_mask


def save_images(img: np.ndarray, 
                OUTPUT_DIRECTORY: str,
                intersection_mask: np.ndarray, 
                expert_annotation_not_reviewed : torch.Tensor,
                expert_annotation_reviewed : torch.Tensor) -> None:
    """
    Saves and displays annotated images, comparing expert and model annotations.

    Parameters:
        img (np.ndarray): The original image in BGR format.
        OUTPUT_DIRECTORY (str): Directory where the annotated image will be saved.
        intersection_mask (np.ndarray): Mask showing intersection areas between model and expert annotations.
        expert_annotation_not_reviewed (torch.Tensor): Expert's annotation mask (not reviewed) as a tensor.
        expert_annotation_reviewed (torch.Tensor): Expert's annotation mask (reviewed) as a tensor.

    This function generates three side-by-side plots:
    1. **Expert Annotation - Not Reviewed**: Displays the expert's initial annotation contours overlaid on the original image in magenta.
    2. **Expert Annotation - Reviewed**: Shows the reviewed expert annotation contours overlaid on the original image in green.
    3. **Model Annotation**: Displays model-detected contours in red on the original image, highlighting areas that intersect with the expert annotations.

    The annotated image is saved as a PNG file in the specified directory, with a filename derived from the original image's name.
    """

    filename_simple = filename.split("/")[-1].split(".")[0]

    plt.figure(figsize=(24, 8)) 
    
    expert_annotation_not_reviewed = expert_annotation_not_reviewed.numpy().astype(np.uint8)  
    img_copy = img.copy()
    if expert_annotation_not_reviewed.any():
        contours, _ = cv2.findContours(expert_annotation_not_reviewed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img_copy, contours, -1, (255, 0, 255), thickness=3) 

    plt.subplot(1, 3, 1)
    plt.title('Expert Annotation - Not Reviewed')
    plt.imshow(cv2.cvtColor(img_copy, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.gca().set_aspect('equal', adjustable='box')
    
    expert_annotation_reviewed = expert_annotation_reviewed.numpy().astype(np.uint8)  
    img_copy = img.copy()
    if expert_annotation_reviewed.any():
        contours, _ = cv2.findContours(expert_annotation_reviewed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img_copy, contours, -1, (0, 255, 0), thickness=3) 

    plt.subplot(1, 3, 2)
    plt.title('Expert Annotation - Reviewed')
    plt.imshow(cv2.cvtColor(img_copy, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.gca().set_aspect('equal', adjustable='box')

    model_img_copy = img.copy()
    if intersection_mask.any():  
        contours, _ = cv2.findContours(intersection_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(model_img_copy, contours, -1, (0, 0, 255), thickness=3)  

    plt.subplot(1, 3, 3)
    plt.title('Model Annotation')
    plt.imshow(cv2.cvtColor(model_img_copy, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.gca().set_aspect('equal', adjustable='box')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIRECTORY + f"/densidade_{filename_simple}.png")
    plt.show()
    plt.close()
        

## Processamento das imagens, salvando os resultados

In [ ]:
for json_file in glob.glob(f"{INPUT_DIRECTORY}/*.json"):

    filename = f"{IMAGE_DIRECTORY}/{json_file.split('/')[-1].split('.')[0]}.jpg"
    img, base_height, base_width = read_image(filename)
    
    if img is None:
        continue

    duct_mask, duct_mask_tensor = prediction_duct(img)
    coral_mask, coral_mask_tensor = prediction_coral(img)

    expert_annotation_reviewed = read_annotation_file(json_file)
    expert_annotation_not_reviewed = read_annotation_file(f"{IMAGE_DIRECTORY}/{json_file.split('/')[-1]}")

    intersection_mask = ((coral_mask_tensor & duct_mask_tensor).int()).cpu().numpy().astype(np.uint8)

    save_images(img, OUTPUT_DIRECTORY, intersection_mask, expert_annotation_not_reviewed, expert_annotation_reviewed)